In [1]:
import pandas as pd

fgkm = pd.read_csv('cf_data/pass2022_fgkm.csv')
mm   = pd.read_csv('cf_data/pass2022_mm.csv')
wdm  = pd.read_csv('cf_data/pass2022_wdm.csv')
lcgen = pd.read_csv('cf_data/dr3_xmatch_from_lcgen.csv')
moca = pd.read_csv('cf_data/mocadb.csv')

In [5]:
pass2022 = pd.concat([fgkm, mm, wdm], ignore_index=True)
pass2022.drop(columns=['age_gyr_cf', 'age_gyr_upper', 'age_gyr_lower'], inplace = True)

In [8]:
pass2022.drop('bprp0', axis=1, inplace=True)

In [26]:
pass2022.to_csv('cf_data/pass2022.csv', index=False)

In [25]:
pass2022[pass2022['flag_high_ruwe']] 

,name,source_id,role,binary_type,mass_msun,prot_days,parallax,bp_rp,ebpminrp_gspphot,ruwe,flag_high_ruwe
0,G 271-110,2478001486169801216,primary,FGKM,0.31,1.060,41.698927,2.879617,NaN,1.708186,True
2,LP 99-392,1603272950424941440,primary,FGKM,0.31,1.196,51.346175,2.776622,0.0000,1.356397,True
4,LP 128-32,837177915851182080,primary,FGKM,0.32,6.500,37.888516,2.794552,0.0000,1.414515,True
8,LP 876-10,6623351805412369024,primary,FGKM,0.20,0.318,130.270657,3.014723,NaN,1.205662,True
12,G 232-62,2005804153064109056,primary,FGKM,0.27,59.000,45.625263,2.851560,0.0000,1.232156,True
14,2MASS J22562702+7600101,2281128259862423680,primary,FGKM,0.19,2.510,40.764760,2.960724,0.0000,1.229958,True
16,2MASS J05363846+1117487,3339914973377653760,primary,FGKM,0.29,5.300,87.345270,2.813897,NaN,1.226539,True
18,HD 183870 B,4187836934609063552,primary,FGKM,0.24,2.320,56.587143,2.867911,0.0000,1.258369,True
20,TIC 22819180,1500827225817637504,NaN,MM,0.18,0.617,42.350580,3.193011,0.0000,1.453971,True
21,TIC 22819191,1500826882220275712,NaN,MM,0.14,0.787,42.323982,3.478862,NaN,1.443001,True


In [16]:
pass2022 = pass2022[~pass2022['name'].isin(['GJ 166 C', 'GJ 166 A', 'GJ 166 B'])]

In [33]:
pass2022['parallax'].describe()

count     68.000000
mean      57.240028
std       34.987399
min       23.647692
25%       30.873231
50%       44.116618
75%       74.589763
max      181.273007
Name: parallax, dtype: float64

In [27]:
from astroquery.gaia import Gaia

job = Gaia.launch_job("""
    SELECT *
    FROM gaiadr3.gaia_source
    WHERE source_id = 837178843564106112
""")

result = job.get_results()
print(result['source_id', 'parallax', 'ruwe', 'bp_rp', 'ebpminrp_gspphot'])

The archive is unstable and may perform below expectations. Please avoid launching intense Python query showers. Please contact the Gaia helpdesk in case of questions (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk). Workaround solutions for the issues following the recent infrastructure upgrade: https://www.cosmos.esa.int/web/gaia/news#WorkaroundArchive
    source_id      parallax ruwe   bp_rp   ebpminrp_gspphot
                     mas            mag          mag       
------------------ -------- ---- --------- ----------------
837178843564106112       --   -- 1.1107507               --


In [29]:
from astroquery.simbad import Simbad
from astroquery.gaia import Gaia

Simbad.add_votable_fields('ra', 'dec')

for name in ['LP 12-72', 'LP 12-90']:
    result = Simbad.query_object(name)
    if result is None:
        print(f"{name}: not found in SIMBAD")
        continue
    ra, dec = result['ra'][0], result['dec'][0]
    print(f"{name}: RA={ra}, DEC={dec}")
    
    job = Gaia.launch_job(f"""
        SELECT source_id, parallax, ruwe, bp_rp, ebpminrp_gspphot
        FROM gaiadr3.gaia_source
        WHERE CONTAINS(
            POINT('ICRS', ra, dec),
            CIRCLE('ICRS', {ra}, {dec}, 0.00139)
        ) = 1
    """)
    print(job.get_results())

LP 12-72: RA=349.85195480385, DEC=79.00106057806
     source_id           parallax         ruwe     bp_rp   ebpminrp_gspphot
                           mas                      mag          mag       
------------------- ------------------ --------- --------- ----------------
2282846074981011328 52.839692674597124 1.2338614 2.7040024              0.0
LP 12-90: RA=350.72427896386, DEC=78.79404648464
     source_id          parallax        ruwe     bp_rp   ebpminrp_gspphot
                          mas                     mag          mag       
------------------- ---------------- --------- --------- ----------------
2282790893241302144 52.8343025839604 1.2345679 3.7322187              0.0


In [30]:
new_rows = [
    {
        'name': 'LP 12-72',
        'source_id': 2282846074981011328,
        'role': '',
        'binary_type': 'MM',
        'mass_msun': 0.53,
        'prot_days': 1.050,
        'parallax': 52.839692674597124,
        'bp_rp': 2.7040024,
        'ebpminrp_gspphot': 0.0,
        'ruwe': 1.2338614,
        'flag_high_ruwe': True
    },
    {
        'name': 'LP 12-90',
        'source_id': 2282790893241302144,
        'role': '',
        'binary_type': 'MM',
        'mass_msun': 0.16,
        'prot_days': 1.044,
        'parallax': 52.8343025839604,
        'bp_rp': 3.7322187,
        'ebpminrp_gspphot': 0.0,
        'ruwe': 1.2345679,
        'flag_high_ruwe': True
    }
]

import pandas as pd
pass2022 = pd.concat([pass2022, pd.DataFrame(new_rows)], ignore_index=True)

In [31]:
pass2022.to_csv('cf_data/pass2022.csv', index=False)    

In [49]:
pass2022 = pass2022.drop(columns=['age_gyr_cf', 'age_gyr_upper', 'age_gyr_lower'], errors='ignore')

fgkm_ages = pd.read_csv('cf_data/pass2022_fgkm.csv')[['source_id', 'age_gyr_cf', 'age_gyr_upper', 'age_gyr_lower']].drop_duplicates(subset='source_id')

pass2022 = pass2022.merge(fgkm_ages, on='source_id', how='left')

In [47]:
pass2022 = pass2022.drop(columns=[c for c in pass2022.columns if c.startswith('age_gyr')])

In [55]:
pass2022.to_csv('cf_data/pass2022.csv', index=False)

In [52]:
pass2022['bprp0'] = pass2022['bp_rp'] - pass2022['ebpminrp_gspphot'].fillna(0)

In [53]:
pass2022

,name,source_id,role,binary_type,mass_msun,prot_days,parallax,bp_rp,ebpminrp_gspphot,ruwe,flag_high_ruwe,bprp0,age_gyr_cf,age_gyr_upper,age_gyr_lower
0,G 271-110,2478001486169801216,primary,FGKM,0.31,1.060,41.698927,2.879617,NaN,1.708186,True,2.879617,0.270778,0.187579,0.127903
1,HD 10008,2477815222028038272,companion,FGKM,NaN,7.100,41.563621,0.970934,0.0,0.967905,False,0.970934,0.270778,0.187579,0.127903
2,LP 99-392,1603272950424941440,primary,FGKM,0.31,1.196,51.346175,2.776622,0.0,1.356397,True,2.776622,0.321519,0.063911,0.060882
3,HD 139477,1603267143629157120,companion,FGKM,NaN,11.200,51.222192,1.292188,0.0,0.968590,False,1.292188,0.321519,0.063911,0.060882
4,LP 128-32,837177915851182080,primary,FGKM,0.32,6.500,37.888516,2.794552,0.0,1.414515,True,2.794552,0.225878,0.083601,0.070191
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,GJ 754.1 A,4201781696994073472,companion,WDM,NaN,NaN,95.175778,0.036789,NaN,1.011426,False,0.036789,NaN,NaN,NaN
66,GJ 169.1 B,470826482637310848,primary,WDM,0.31,NaN,181.273007,0.442300,NaN,0.926382,False,0.442300,NaN,NaN,NaN
67,GJ 169.1 A,470826482637310080,companion,WDM,NaN,NaN,181.243839,2.895883,NaN,1.218251,True,2.895883,NaN,NaN,NaN
68,LP 12-72,2282846074981011328,,MM,0.53,1.050,52.839693,2.704002,0.0,1.233861,True,2.704002,NaN,NaN,NaN


In [54]:
pass2022[pass2022['binary_type'] == 'FGKM']

,name,source_id,role,binary_type,mass_msun,prot_days,parallax,bp_rp,ebpminrp_gspphot,ruwe,flag_high_ruwe,bprp0,age_gyr_cf,age_gyr_upper,age_gyr_lower
0,G 271-110,2478001486169801216,primary,FGKM,0.31,1.060,41.698927,2.879617,NaN,1.708186,True,2.879617,0.270778,0.187579,0.127903
1,HD 10008,2477815222028038272,companion,FGKM,NaN,7.100,41.563621,0.970934,0.0,0.967905,False,0.970934,0.270778,0.187579,0.127903
2,LP 99-392,1603272950424941440,primary,FGKM,0.31,1.196,51.346175,2.776622,0.0,1.356397,True,2.776622,0.321519,0.063911,0.060882
3,HD 139477,1603267143629157120,companion,FGKM,NaN,11.200,51.222192,1.292188,0.0,0.968590,False,1.292188,0.321519,0.063911,0.060882
4,LP 128-32,837177915851182080,primary,FGKM,0.32,6.500,37.888516,2.794552,0.0,1.414515,True,2.794552,0.225878,0.083601,0.070191
5,HD 93811,837178843564106112,companion,FGKM,NaN,8.500,NaN,1.110751,NaN,NaN,False,1.110751,0.225878,0.083601,0.070191
6,LP 876-10,6623351805412369024,primary,FGKM,0.20,0.318,130.270657,3.014723,NaN,1.205662,True,3.014723,0.219503,0.046155,0.034642
7,HD 216803,6604147121141267712,companion,FGKM,NaN,10.100,131.552509,1.340312,0.0,0.924698,False,1.340312,0.219503,0.046155,0.034642
8,LSPM J0849+0329W,581103581886377728,primary,FGKM,0.23,4.400,32.950051,2.915384,NaN,1.083455,False,2.915384,4.030931,0.532371,0.504078
9,HD 75302,581100382135253760,companion,FGKM,NaN,16.400,32.977697,0.850505,0.0,1.003923,False,0.850505,4.030931,0.532371,0.504078
